In [0]:
import os, sys
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.append("/Workspace" + os.path.dirname(notebook_path))

from config_prod import window_days, lag_days
from si1_functions import prepare_orders, prepare_pit_reviews, create_dense_calendar, calculate_pit_metrics


# ============================================================
# PIPELINE EXECUTION
# ============================================================
 
# 1. Read source tables
fact_orders = spark.read.table("marketplace_olist.gold.fact_orders")
fact_order_items = spark.read.table("marketplace_olist.gold.fact_order_items")
fact_reviews = spark.read.table("marketplace_olist.gold.fact_reviews")
 
# 2. Prepare Data and State Ledgers
pit_orders = prepare_orders(fact_orders, fact_order_items)
pit_reviews = prepare_pit_reviews(fact_reviews, fact_orders)
dense_calendar = create_dense_calendar(pit_orders, fact_order_items, window_days, lag_days)
 
# 3. Calculate final metrics dynamically
df_fact_seller_metrics = calculate_pit_metrics(dense_calendar, pit_orders, pit_reviews, window_days)

In [0]:
# Save to Gold Medallion Layer
df_fact_seller_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("marketplace_olist.gold.fact_seller_metrics")